In [2]:
from pathlib import Path
import sys
project_root=Path.cwd().parents[1]
sys.path.append(str(project_root))

In [7]:
from typing import Any

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    BaseMessage,
    AIMessage
)

from src.query_rewriter.base import BaseQueryRewriter
from src.query_rewriter.models import (
    ChatMessage,
    QueryRewriteRequest,
    ReformulationResult
)

from src.query_rewriter.prompts import REFORMULATION_SYSTEM_PROMPT
from src.query_rewriter.utils import (llm_util, message_util)

In [ ]:
class Reformulator(BaseQueryRewriter):

    def __init__(self, llm):
        self.llm=llm
    
    def _build_messages( self, request: QueryRewriteRequest ) ->list :

        messages: list[BaseMessage] =[ SystemMessage( content = REFORMULATION_SYSTEM_PROMPT )]

        messages.extend(message_util.build_messages(history=request.history))
        
        messages.append(
            HumanMessage(
                content=f"""
Latest User Query:

{request.query}

Rewrite the above query into a standalone search query.
"""
            )
        )

        return messages
    
    
    def _parse_response(self, response:AIMessage) ->str:
        if not isinstance(response.content, str):
            raise TypeError("Expected text response from the LLM.")

        return response.content.strip()
    
    def _validate(self, rewritten_query:str)->str:
        if not rewritten_query:
            raise ValueError(" Empty reformulated query")
        return rewritten_query
    
    def rewrite(self, request: QueryRewriteRequest)-> ReformulationResult:

        messages=self._build_messages(request=request)
        response=llm_util.invoke_llm(llm=self.llm, messages=messages)

        rewritten_query=self._parse_response(response)
        rewritten_query=self._validate(rewritten_query)

        return ReformulationResult( request.query, reformulated_query=rewritten_query)
